<a href="https://colab.research.google.com/github/Aytijhya/Seq-changepoint-localization/blob/main/sentiment_CPL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib_inline.backend_inline
from scipy.stats import ks_1samp, uniform, norm, gaussian_kde
import torch
from torchvision.datasets import MNIST
from torchvision import transforms
from torchvision.models import ResNet18_Weights
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from datasets import load_dataset
from joblib import Parallel, delayed

from tqdm import tqdm

import timm

np.random.seed(40)

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")
# plt.style.use("math.mplstyle") # Commented out as 'math.mplstyle' was not found.

plt.rcParams.update({"axes.labelsize": 14})
plt.rcParams.update({"xtick.labelsize": 14})
plt.rcParams.update({"ytick.labelsize": 14})
plt.rcParams.update({"legend.fontsize": 14})
plt.rcParams.update({"axes.titlesize": 16})

In [ ]:
def get_pretrained_sentiment_model(device="cpu"):
    model_name = "distilbert-base-uncased-finetuned-sst-2-english"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)

    model.to(device)
    model.eval()
    return model, tokenizer


def generate_sentiment_dataset(length, changepoint, dataset_name="sst2"):
    dataset = load_dataset(dataset_name)

    train_data = dataset["train"]
    positive_texts = [item["sentence"] for item in train_data if item["label"] == 1]
    negative_texts = [item["sentence"] for item in train_data if item["label"] == 0]

    random.shuffle(positive_texts)
    random.shuffle(negative_texts)

    n1 = changepoint + 1
    n2 = length - n1

    if n1 > len(positive_texts) or n2 > len(negative_texts):
        raise ValueError("Insufficient texts for the specified length and changepoint.")

    texts_before = positive_texts[:n1]
    texts_after = negative_texts[:n2]

    texts = texts_before + texts_after
    true_labels = [1] * n1 + [0] * n2

    return texts, true_labels


def generate_mixed_sentiment_dataset(length, changepoint, dataset_name="sst2", pos_pre=0.75, pos_post=0.25):
    dataset = load_dataset(dataset_name)

    train_data = dataset["train"]
    positive_texts = [item["sentence"] for item in train_data if item["label"] == 1]
    negative_texts = [item["sentence"] for item in train_data if item["label"] == 0]

    random.shuffle(positive_texts)
    random.shuffle(negative_texts)

    n_pre = changepoint + 1
    n_post = length - n_pre

    n_pos_pre = int(n_pre * pos_pre)
    n_neg_pre = n_pre - n_pos_pre

    n_pos_post = int(n_post * pos_post)
    n_neg_post = n_post - n_pos_post

    if n_pos_pre + n_pos_post > len(positive_texts) or n_neg_pre + n_neg_post > len(
        negative_texts
    ):
        raise ValueError(
            "Insufficient texts for the specified distribution and length."
        )

    pre_pos_texts = positive_texts[:n_pos_pre]
    pre_neg_texts = negative_texts[:n_neg_pre]
    pre_texts = pre_pos_texts + pre_neg_texts
    pre_labels = [1] * n_pos_pre + [0] * n_neg_pre

    pre_combined = list(zip(pre_texts, pre_labels))
    random.shuffle(pre_combined)
    pre_texts, pre_labels = zip(*pre_combined)

    post_pos_texts = positive_texts[n_pos_pre : n_pos_pre + n_pos_post]
    post_neg_texts = negative_texts[n_neg_pre : n_neg_pre + n_neg_post]
    post_texts = post_pos_texts + post_neg_texts
    post_labels = [1] * n_pos_post + [0] * n_neg_post

    post_combined = list(zip(post_texts, post_labels))
    random.shuffle(post_combined)
    post_texts, post_labels = zip(*post_combined)

    texts = list(pre_texts) + list(post_texts)
    true_labels = list(pre_labels) + list(post_labels)
    print(sum(pre_labels))
    print(sum(post_labels))
    return texts, true_labels

def generate_mixed_sentiment_null_dataset(length, dataset_name="sst2", pos_pre = 0.6):
    dataset = load_dataset(dataset_name)

    train_data = dataset["train"]
    positive_texts = [item["sentence"] for item in train_data if item["label"] == 1]
    negative_texts = [item["sentence"] for item in train_data if item["label"] == 0]

    random.shuffle(positive_texts)
    random.shuffle(negative_texts)

    n_pre = length


    n_pos_pre = int(n_pre * pos_pre)
    n_neg_pre = n_pre - n_pos_pre


    if n_pos_pre > len(positive_texts) or n_neg_pre > len(
        negative_texts
    ):
        raise ValueError(
            "Insufficient texts for the specified distribution and length."
        )

    pre_pos_texts = positive_texts[:n_pos_pre]
    pre_neg_texts = negative_texts[:n_neg_pre]
    pre_texts = pre_pos_texts + pre_neg_texts
    pre_labels = [1] * n_pos_pre + [0] * n_neg_pre

    pre_combined = list(zip(pre_texts, pre_labels))
    random.shuffle(pre_combined)
    pre_texts, pre_labels = zip(*pre_combined)


    texts = list(pre_texts)
    true_labels = list(pre_labels)

    return texts, true_labels


def predict_sentiment(model, tokenizer, text, device="cpu"):
    inputs = tokenizer(
        text, return_tensors="pt", padding=True, truncation=True, max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1).cpu()
        predicted = probs.argmax(dim=1).item()

    return (predicted, probs.squeeze())

In [ ]:
def compute_sequential_sentiment_scores(texts, device="cpu", batch_size=4, max_length=128):
    model_name = "distilbert-base-uncased-finetuned-sst-2-english"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)
    model.to(device)
    model.eval()

    n = len(texts)
    preds = []
    probs_list = []
    cusum = 1
    with torch.no_grad():
        for start in tqdm(range(0, n, batch_size)):
            end = min(start + batch_size, n)
            batch_texts = texts[start:end]
            inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True, max_length=max_length).to(device)
            outputs = model(**inputs)
            batch_probs = torch.softmax(outputs.logits, dim=1).detach().cpu()
            probs_list.append(batch_probs)

            s = batch_probs.argmax(dim=1).tolist()
            preds.extend(s)

            sum_s = sum(s)

            q = 0.6
            p = 0.4
            # Calculate log-likelihood ratio for CUSUM to handle edge cases more robustly.
            log_likelihood_ratio = sum_s * np.log(p / q) + (batch_size - sum_s) * np.log((1 - p) / (1 - q))


            likelihood_ratio = np.exp(log_likelihood_ratio)
            cusum = cusum * max(1, likelihood_ratio)

            if likelihood_ratio < 1:
                that = len(preds)
            if cusum > 10000:
                print(f"cusum: {cusum}")
                return preds, that
        else:
            return 0,0

In [ ]:
def run_mixed_sentiment_simulation(n, xi, device="cpu"):
    texts, true_labels = generate_mixed_sentiment_dataset(n, xi, dataset_name="sst2", pos_pre=0.75, pos_post=0.25)
    predictions, that = compute_sequential_sentiment_scores(texts, device=device, batch_size=6, max_length=128)
    return predictions, that, texts
n = 1000
xi = 500
p_values, that, texts = run_mixed_sentiment_simulation(n, xi)

In [ ]:
len(p_values)

In [ ]:
that

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
np.random.seed(40)
# Initialize 'r' as a list of lists to prevent IndexError
# The outer list has 100 elements (for 100 iterations),
# and each inner list has a length of len(p_values).
r = [[0] * len(p_values) for _ in range(100)]

for i in range(20):
  nulltexts, null_labels = generate_mixed_sentiment_null_dataset(len(p_values), dataset_name="sst2", pos_pre=0.6)
  predictions, t = compute_sequential_sentiment_scores(nulltexts, device=device, batch_size=16, max_length=128)
  if(t == 0):
    r[i]=[1] * len(p_values) # This line will now correctly assign to r[i]
  else:
    for j in range(len(p_values)):
      r[i][j]=len(predictions)>j


In [ ]:
def compute_Mt(X, p, q):
    X = np.array(X, dtype=float)
    tau = len(X)

    # 1. Log-Likelihood Ratios
    L = X * np.log(q / p) + (1 - X) * np.log((1 - q) / (1 - p))

    # 2. Term 1 Logic: max_{1 <= i <= t-1} sum_{j=i}^{t-1} Lj
    # We want to subtract the minimum of the previous cumulative sums.
    S = np.cumsum(L)
    S_with_zero = np.insert(S, 0, 0.0) # Length is tau + 1

    term1_log = np.full(tau, -np.inf)
    if tau > 1:
        # running_min[k] is min(S_0, ..., S_k)
        running_min = np.minimum.accumulate(S_with_zero)
        # For t=2 (index 1), we need S[0] - running_min[0]
        # For t=tau (index tau-1), we need S[tau-2] - running_min[tau-2]
        # S[:-1] is length tau-1, running_min[:-2] is length tau-1
        term1_log[1:] = S[:-1] - running_min[:-2]

    # 3. Term 2 Logic: max_{t <= i <= tau} sum_{j=t}^{i} -Lj
    L_neg = -L
    S_neg = np.cumsum(L_neg)
    S_neg_with_zero = np.insert(S_neg, 0, 0.0) # Length is tau + 1

    # Suffix max: max(S_neg[t], ..., S_neg[tau])
    # We ignore the 0.0 at the start for the suffix max calculation
    suffix_max = np.maximum.accumulate(S_neg[::-1])[::-1] # Length is tau

    # term2_log[t-1] = suffix_max[t-1] - S_neg_zero[t-1]
    # Both suffix_max and S_neg_with_zero[:-1] are length tau.
    term2_log = suffix_max - S_neg_with_zero[:-1]

    # 4. Final Result
    Mt_log = np.maximum(term1_log, term2_log)
    return np.exp(Mt_log)

In [ ]:
import numpy as np # Ensure numpy is imported for array operations

# Initialize 'test' list with zeros to hold results, matching the length of p_values
test = [0.0] * len(p_values)
p=0.6
q=0.4
confset=[]


# Pre-calculate log terms to avoid redundant calculations inside the loop
log_p_over_q = np.log(p / q)
log_1_minus_p_over_1_minus_q = np.log((1 - p) / (1 - q))
log_q_over_p = np.log(q / p)
log_1_minus_q_over_1_minus_p = np.log((1 - q) / (1 - p))

for i in range(len(p_values)):
    if i < that:
        # Convert the list slice to a NumPy array for element-wise operations
        segment = np.array(p_values[i:that])
        test[i] = np.sum(segment * log_p_over_q + (1 - segment) * log_1_minus_p_over_1_minus_q)
    if i == that:
        test[i] = 0
    elif i > that:
        # Convert the list slice to a NumPy array for element-wise operations
        # This calculates a sum from 'that' onwards.
        segment = np.array(p_values[that:i])
        test[i] = np.sum(segment * log_q_over_p + (1 - segment) * log_1_minus_q_over_1_minus_p)

    if test[i] < np.log(2*20 / (0.05*(sum(sublist[i] for sublist in r if len(sublist) > 1)))):
        confset.append(i)

In [ ]:
p=0.6
q=0.4
confset=[]
test = compute_Mt(p_values, p, q)
for i in range(len(p_values)):
  # Modified the condition to explicitly check if 'i' is a valid index for 'sublist'
  # before attempting to access sublist[i].
  # This prevents IndexError if any sublist in 'r' is shorter than expected.
  if test[i]<2*20 / (0.1*(sum(sublist[j] for sublist in r if j < len(sublist)))):
    confset.append(i)

In [ ]:
confset

In [ ]:
texts[480:520]

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import ticker # Import ticker for minor locator

start_index = 470 # This was the start_index in the original plot for this cell
end_index = len(p_values) + 1 # This was the end_index in the original plot for this cell
selected_texts_original = texts[start_index:end_index]
indices = list(range(start_index, end_index))

# Truncate each text to a maximum of 15 words
truncated_texts = []
for text_content in selected_texts_original:
    words = text_content.split()
    truncated = ' '.join(words[:14])
    if len(words) > 14:
        truncated += '...'
    truncated_texts.append(truncated)
selected_texts = truncated_texts # Use truncated texts for plotting

# Calculate dynamic figure dimensions
# Figure width needs to accommodate many labels on x-axis
fig_width = max(2, len(selected_texts) * 0.23) # Reduced multiplier to make it denser
fig_height = max(5, max(len(t) for t in selected_texts) * 0.088) # Adjusted multiplier for height based on truncated texts

plt.figure(figsize=(fig_width, fig_height))
ax = plt.gca()

# Set MAJOR x-axis ticks to show only even numbers for labels
even_indices = [idx for idx in indices if idx % 2 == 0]
ax.set_xticks(even_indices)
ax.tick_params(axis='x', rotation=4) # Rotate x-axis labels if they are dense

# Set MINOR x-axis ticks to show all integer indices for grid lines
ax.xaxis.set_minor_locator(ticker.MultipleLocator(1)) # Place minor ticks at every integer

# Remove y-axis entirely as per your request
ax.set_yticks([])
ax.set_ylabel('')

# Iterate and place each text vertically above its index
# y_offset determines how high above the x-axis the text starts
y_offset = 0.05 # Small offset to place text just above the x-axis line
for i, text_content in enumerate(selected_texts):
    # The x-coordinate is the absolute index
    # The y-coordinate is a small offset from the bottom of the plot
    # 'rotation=90' makes the text vertical
    # 'va="bottom"' aligns the bottom of the text with the y-coordinate
    # 'ha="center"' centers the text horizontally over the x-tick
    ax.text(indices[i], y_offset, text_content, rotation=90, va='bottom', ha='center', fontsize=10)

# Add red bold dots for indices in confset
# Filter confset to only include indices within the current plot range
confset_in_range = [idx for idx in confset if start_index <= idx < end_index]
if confset_in_range:
    # Plot dots slightly above the x-axis (e.g., at y=0.01) to ensure full visibility
    ax.scatter(confset_in_range, [0.01] * len(confset_in_range), color='red', marker='o', s=100, zorder=5) # zorder to ensure visibility

# Add vertical black line at x=500
ax.axvline(x=500, color='black', linestyle='-', linewidth=2)

# Add vertical dotted red line at x='that'
ax.axvline(x=that, color='red', linestyle=':', linewidth=2)

# Set x-axis label
plt.xlabel('Time', fontsize=12)
#plt.title(f'Text Segments Displayed Vertically by Index {start_index} to {end_index-1}', fontsize=14)

# Adjust x-axis limits to ensure all text and ticks are visible
ax.set_xlim(start_index - 0.5, end_index - 0.5)

# Adjust y-axis limits to accommodate the longest rotated text
# This might need fine-tuning based on average text length and font size
max_text_display_length = max(len(t) for t in selected_texts) * 0.015 # Estimate based on font size and char width
ax.set_ylim(0, y_offset + max_text_display_length + 0.1)

# Enable grid lines for both major and minor ticks with dashed style
ax.grid(True, which='both', axis='x', linestyle='--', alpha=0.7) # Grid lines on both major and minor ticks

plt.tight_layout() # Adjust layout to prevent labels from overlapping

# Save the plot as a PNG file
plt.savefig('text_segments_plot.png')
plt.show()